In [1]:
#!wqms-summaries generate-tasks --output-dir="test-summaries" "s3://deafrica-water-quality-dev/historical-extent-rasters/"

In [2]:
# Mimic generate-tasks
import json
with open("test-summaries/tasks", "w") as file:
    file.write(json.dumps(['s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x211y076.tif']))

This notebook demonstrates how the per water body summaries are generated for a single tile. 

In [3]:
%load_ext water_quality.magics

In [4]:
import os
# This ensures the database is a sqlite file database
os.environ["TestingMode"] = "True"
bool(os.environ.get("TestingMode", None))

True

In [5]:
import json
import logging
import sys

import click
import pandas as pd
import rioxarray
import xarray as xr
from datacube import Datacube
from waterbodies.db import get_waterbodies_engine

from water_quality.io import check_directory_exists, get_filesystem, join_url
from water_quality.logs import setup_logging
from water_quality.summaries.summary import (
    add_water_quality_observations_to_db,
)
from water_quality.tasks import split_tasks

In [6]:
tasks="test-summaries/tasks"
waterbodies_to_filter="test-summaries/waterbodies_for_vector_processing"
max_parallel_steps=1
worker_idx=0
overwrite=True
log="INFO"

In [7]:
log_level = getattr(logging, log.upper())
_log = setup_logging(log_level)

In [8]:
fs = get_filesystem(waterbodies_to_filter, anon=False)
with fs.open(waterbodies_to_filter, "r") as file:
    uids_to_exclude = set(sorted(json.load(file)))

In [9]:
fs = get_filesystem(tasks, anon=False)
with fs.open(tasks, "r") as file:
    all_tasks = sorted(json.load(file))

tasks_to_run = split_tasks(all_tasks, max_parallel_steps, worker_idx)

if not tasks_to_run:
    _log.warning(f"Worker {worker_idx} has no tasks to process. Exiting.")
    sys.exit(0)

_log.info(f"Worker {worker_idx} processing {len(tasks_to_run)} tasks")

2026-03-13 20:11:06,221 __main__ [INFO]: Worker 0 processing 1 tasks


In [11]:
tasks_to_run

('s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x211y076.tif',)

In [12]:
dc = Datacube(app="process_raster_tasks")
measurements = [
    "fai",
    "ndvi",
    "hue",
    "owt",
    "chla",
    "tsi",
    "tsm",
    "st_max",
    "st_median",
    "st_min",
    "water_mask",
]
product = "wq_annual"
dask_chunks = {"x": 800, "y": 800}
# m2_per_km2 = 1_000_000
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

engine = get_waterbodies_engine()

2026-03-13 20:11:33,314 alembic.runtime.plugins [INFO]: setup plugin alembic.autogenerate.schemas
2026-03-13 20:11:33,315 alembic.runtime.plugins [INFO]: setup plugin alembic.autogenerate.tables
2026-03-13 20:11:33,315 alembic.runtime.plugins [INFO]: setup plugin alembic.autogenerate.types
2026-03-13 20:11:33,316 alembic.runtime.plugins [INFO]: setup plugin alembic.autogenerate.constraints
2026-03-13 20:11:33,316 alembic.runtime.plugins [INFO]: setup plugin alembic.autogenerate.defaults
2026-03-13 20:11:33,316 alembic.runtime.plugins [INFO]: setup plugin alembic.autogenerate.comments


In [13]:
engine.url

sqlite+pysqlite:////tmp/test_waterbodies.db

In [16]:
failed_tasks = []
for idx, cog_path in enumerate(tasks_to_run):
    _log.info(
        f"Processing historical extent COG {idx + 1} of {len(tasks_to_run)}: {cog_path} "
    )
    break

2026-03-13 20:11:53,220 __main__ [INFO]: Processing historical extent COG 1 of 1: s3://deafrica-water-quality-dev/historical-extent-rasters/historical_extent_x211y076.tif 


In [19]:
extent_da = rioxarray.open_rasterio(cog_path).squeeze()
extent_da

<xarray.DataArray (y: 9600, x: 9600)> Size: 737MB
[92160000 values with dtype=int64]
Coordinates:
  * y            (y) float64 77kB -5.0 -15.0 -25.0 ... -9.598e+04 -9.6e+04
  * x            (x) float64 77kB 2.88e+06 2.88e+06 ... 2.976e+06 2.976e+06
    band         int64 8B 1
    spatial_ref  int64 8B 0
Attributes:
    WB_ID_to_UID:   {"379743": "kxv7txxgy2", "379742": "kxv7twsc0f", "379992"...
    AREA_OR_POINT:  Area
    _FillValue:     0
    scale_factor:   1.0
    add_offset:     0.0

In [20]:
wb_id_to_uid = {
    int(wb_id): uid
    for wb_id, uid in json.loads(
        extent_da.attrs["WB_ID_to_UID"]
    ).items()
}
wb_ids_to_exclude = [
    wb_id
    for wb_id, uid in wb_id_to_uid.items()
    if uid in uids_to_exclude
]
wb_id_to_uid_filtered = {
    wb_id: uid
    for wb_id, uid in wb_id_to_uid.items()
    if wb_id not in wb_ids_to_exclude
}

assert len(wb_id_to_uid_filtered) == len(wb_id_to_uid) - len(
    wb_ids_to_exclude
)

if len(wb_ids_to_exclude) > 0:
    extent_da = extent_da.where(
        ~extent_da.isin(wb_ids_to_exclude), other=0
    )

extent_da = extent_da.where(extent_da != 0)
extent_da

<xarray.DataArray (y: 9600, x: 9600)> Size: 737MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(9600, 9600))
Coordinates:
  * y            (y) float64 77kB -5.0 -15.0 -25.0 ... -9.598e+04 -9.6e+04
  * x            (x) float64 77kB 2.88e+06 2.88e+06 ... 2.976e+06 2.976e+06
    band         int64 8B 1
    spatial_ref  int64 8B 0
Attributes:
    WB_ID_to_UID:   {"379743": "kxv7txxgy2", "379742": "kxv7twsc0f", "379992"...
    AREA_OR_POINT:  Area
    _FillValue:     0
    scale_factor:   1.0
    add_offset:     0.0

In [21]:
ds = dc.load(
    product=product,
    like=extent_da.odc.geobox,
    measurements=measurements,
    dask_chunks=dask_chunks,
)
ds

<xarray.Dataset> Size: 101GB
Dimensions:      (time: 25, y: 9600, x: 9600)
Coordinates:
  * time         (time) datetime64[ns] 200B 2000-07-01T23:59:59.999999 ... 20...
  * y            (y) float64 77kB -5.0 -15.0 -25.0 ... -9.598e+04 -9.6e+04
  * x            (x) float64 77kB 2.88e+06 2.88e+06 ... 2.976e+06 2.976e+06
    spatial_ref  int32 4B 6933
Data variables:
    fai          (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    ndvi         (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    hue          (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    owt          (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    chla         (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    tsi          (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    tsm          (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    st_max       (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    st_median    (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    st_min       (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
    water_mask   (time, y, x) float32 9GB dask.array<chunksize=(1, 800, 800), meta=np.ndarray>
Attributes:
    crs:           PROJCS["WGS 84 / NSIDC EASE-Grid 2.0 Global",GEOGCS["WGS 8...
    grid_mapping:  spatial_ref

In [24]:
_log.info(
    f"Processing per waterbody statistics for {ds.time.size} years for {len(wb_id_to_uid_filtered)} waterbodies ..."
)
df_drop_columns = ["band", "spatial_ref"]

2026-03-13 20:14:39,221 __main__ [INFO]: Processing per waterbody statistics for 25 years for 91 waterbodies ...


In [25]:
_log.info(
    "Processing water area consistently indicating algae ..."
)
water_mask_count = (
    (~ds["water_mask"].isnull()).groupby(extent_da).sum()
)
fai_count = (~ds["fai"].isnull()).groupby(extent_da).sum()
fai_cover = (
    fai_count / water_mask_count.where(water_mask_count > 0)
) * 100
fai_cover_df = fai_cover.to_dataframe(name="fai_cover").drop(
    columns=df_drop_columns
)

fai_cover_df

2026-03-13 20:14:55,222 __main__ [INFO]: Processing water area consistently indicating algae ...


fai_cover
time                       group              
2000-07-01 23:59:59.999999 379743.0        NaN
                           380010.0        NaN
                           380011.0  15.718157
                           380012.0        NaN
                           380023.0        NaN
...                                        ...
2024-07-01 23:59:59.999999 380111.0   4.040404
                           380112.0  24.521073
                           380113.0  46.666667
                           380114.0  31.746032
                           380115.0  37.777778

[2275 rows x 1 columns]

In [26]:
_log.info(
    "Processing water area consistently indicating vegetation ..."
)
ndvi_count = (~ds["ndvi"].isnull()).groupby(extent_da).sum()
ndvi_cover = (
    ndvi_count / water_mask_count.where(water_mask_count > 0)
) * 100

ndvi_cover_df = ndvi_cover.to_dataframe(name="ndvi_cover").drop(
    columns=df_drop_columns
)
ndvi_cover_df

2026-03-13 20:16:07,837 __main__ [INFO]: Processing water area consistently indicating vegetation ...


ndvi_cover
time                       group               
2000-07-01 23:59:59.999999 379743.0         NaN
                           380010.0         NaN
                           380011.0       100.0
                           380012.0         NaN
                           380023.0         NaN
...                                         ...
2024-07-01 23:59:59.999999 380111.0       100.0
                           380112.0       100.0
                           380113.0       100.0
                           380114.0       100.0
                           380115.0       100.0

[2275 rows x 1 columns]

In [27]:
annual_quantile_measurements = [
    "hue",
    "owt",
    "chla",
    "tsi",
    "tsm",
    "st_max",
    "st_median",
    "st_min",
]
# Sanity check
assert set(annual_quantile_measurements).issubset(
    set(measurements)
)
assert set(annual_quantile_measurements).issubset(
    set(list(ds.data_vars))
)

In [28]:
quantiles_df_to_merge = []
for measurement in annual_quantile_measurements:
    _log.info(
        f"Processing per waterbody quantiles for the {measurement} variable"
    )
    with xr.set_options(use_flox=False):
        quantiles_da = (
            ds[measurement].groupby(extent_da).quantile(quantiles)
        )
    quantiles_df = quantiles_da.to_dataframe().unstack("quantile")
    quantiles_df.columns = [
        f"{measurement}_q{q}".replace(".", "_")
        for _, q in quantiles_df.columns
    ]
    quantiles_df_to_merge.append(quantiles_df)

2026-03-13 20:16:59,470 __main__ [INFO]: Processing per waterbody quantiles for the hue variable
2026-03-13 20:18:18,687 __main__ [INFO]: Processing per waterbody quantiles for the owt variable
2026-03-13 20:19:27,732 __main__ [INFO]: Processing per waterbody quantiles for the chla variable
2026-03-13 20:20:37,332 __main__ [INFO]: Processing per waterbody quantiles for the tsi variable
2026-03-13 20:21:47,430 __main__ [INFO]: Processing per waterbody quantiles for the tsm variable
2026-03-13 20:22:57,602 __main__ [INFO]: Processing per waterbody quantiles for the st_max variable
2026-03-13 20:24:07,666 __main__ [INFO]: Processing per waterbody quantiles for the st_median variable
2026-03-13 20:25:17,259 __main__ [INFO]: Processing per waterbody quantiles for the st_min variable


In [30]:
per_waterbody_summaries = pd.concat(
    [*quantiles_df_to_merge, fai_cover_df, ndvi_cover_df], axis=1
)
per_waterbody_summaries = (
    per_waterbody_summaries.reset_index().rename(
        columns={"group": "wb_id"}
    )
)

per_waterbody_summaries["uid"] = per_waterbody_summaries[
    "wb_id"
].map(wb_id_to_uid_filtered)

per_waterbody_summaries["obs_id"] = (
    per_waterbody_summaries["time"].dt.strftime("%Y/%m/%d")
    + "_"
    + per_waterbody_summaries["uid"]
)

per_waterbody_summaries = per_waterbody_summaries.rename(
    columns={"time": "date"}
)
cols = per_waterbody_summaries.columns.tolist()
cols.insert(0, cols.pop(cols.index("uid")))
cols.insert(0, cols.pop(cols.index("obs_id")))

per_waterbody_summaries = per_waterbody_summaries[cols].drop(
    columns=["wb_id"]
)
per_waterbody_summaries

,obs_id,uid,date,hue_q0_1,hue_q0_2,hue_q0_3,hue_q0_4,hue_q0_5,hue_q0_6,hue_q0_7,...,st_min_q0_2,st_min_q0_3,st_min_q0_4,st_min_q0_5,st_min_q0_6,st_min_q0_7,st_min_q0_8,st_min_q0_9,fai_cover,ndvi_cover
0,2000/07/01_kxv7txxgy2,kxv7txxgy2,2000-07-01 23:59:59.999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000/07/01_kxvkvdcwb9,kxvkvdcwb9,2000-07-01 23:59:59.999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000/07/01_kxvmqfpr2g,kxvmqfpr2g,2000-07-01 23:59:59.999999,65.075523,68.414290,70.306278,71.614049,72.541687,73.329263,73.961272,...,24.258887,24.335111,24.463787,24.591014,24.719041,24.913462,25.292544,25.862793,15.718157,100.0
3,2000/07/01_kxvnpqqgnb,kxvnpqqgnb,2000-07-01 23:59:59.999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000/07/01_kxvppeum9d,kxvppeum9d,2000-07-01 23:59:59.999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2270,2024/07/01_kxvz3bmur9,kxvz3bmur9,2024-07-01 23:59:59.999999,52.747874,54.442206,55.547493,56.523738,57.331818,58.360995,59.490118,...,27.717723,27.804502,27.841443,27.942921,28.007575,28.097629,28.188004,28.359895,4.040404,100.0
2271,2024/07/01_kxvz90p9qc,kxvz90p9qc,2024-07-01 23:59:59.999999,51.521418,52.260491,53.205862,53.977848,54.642681,55.206715,55.874445,...,24.626146,24.804581,24.956333,25.104448,25.304451,25.510977,25.755476,26.136326,24.521073,100.0
2272,2024/07/01_kxvz9pkd6v,kxvz9pkd6v,2024-07-01 23:59:59.999999,50.606647,50.913123,51.086071,51.499561,51.850805,52.239749,52.444054,...,28.674466,28.899093,29.078790,29.177456,29.299467,29.400494,29.490704,29.606368,46.666667,100.0
2273,2024/07/01_kxvzk8pdhp,kxvzk8pdhp,2024-07-01 23:59:59.999999,57.063499,57.836127,59.352818,152.954459,169.359146,177.449667,179.862949,...,24.194794,24.230059,24.725130,25.571819,25.826429,26.104671,27.287075,28.227490,31.746032,100.0


In [31]:
add_water_quality_observations_to_db(
    water_quality_measures=per_waterbody_summaries,
    engine=engine,
    update_rows=overwrite,
)

2026-03-13 20:27:34,263 water_quality.summaries.summary [INFO]: Found 1638 out of 2275 waterbody observations already in the waterbodies_water_quality table
2026-03-13 20:27:35,867 water_quality.summaries.summary [INFO]: Updating 1638 water quality observations in the waterbodies_water_quality table
2026-03-13 20:27:37,454 water_quality.summaries.summary [INFO]: Inserting 637 water quality observations in the waterbodies_water_quality table


In [32]:
results = pd.read_sql("SELECT * FROM waterbodies_water_quality", con=engine )
results

,obs_id,uid,date,hue_q0_1,hue_q0_2,hue_q0_3,hue_q0_4,hue_q0_5,hue_q0_6,hue_q0_7,...,st_min_q0_2,st_min_q0_3,st_min_q0_4,st_min_q0_5,st_min_q0_6,st_min_q0_7,st_min_q0_8,st_min_q0_9,fai_cover,ndvi_cover
0,2007/07/02_kxv7txxgy2,kxv7txxgy2,2007-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2007/07/02_kxvkvdcwb9,kxvkvdcwb9,2007-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2007/07/02_kxvmqfpr2g,kxvmqfpr2g,2007-07-02,64.113471,65.6211,66.717152,67.866568,69.088661,70.07392,71.233648,...,23.640678,23.755594,23.833941,23.914207,23.981078,24.092568,24.315033,24.723758,20.481928,100.0
3,2007/07/02_kxvnpqqgnb,kxvnpqqgnb,2007-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2007/07/02_kxvppeum9d,kxvppeum9d,2007-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2270,2006/07/02_kxvz3bmur9,kxvz3bmur9,2006-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2271,2006/07/02_kxvz90p9qc,kxvz90p9qc,2006-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.327445,15.497772,15.638831,15.818607,15.989104,16.083138,16.329385,16.829510,100.000000,100.0
2272,2006/07/02_kxvz9pkd6v,kxvz9pkd6v,2006-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2273,2006/07/02_kxvzk8pdhp,kxvzk8pdhp,2006-07-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,17.487070,17.500286,17.520840,17.551329,17.596674,17.619368,17.670775,17.916419,100.000000,100.0
